In [0]:
CREATE WIDGET TEXT academic_year DEFAULT "202526";
CREATE WIDGET TEXT ilr_snapshot DEFAULT "SN06";
CREATE WIDGET TEXT snapshot DEFAULT "06";
CREATE WIDGET TEXT source_catalog DEFAULT "catalog_40_copper_proj_fe_skills_statistics_dev";    
CREATE WIDGET TEXT output_catalog DEFAULT "catalog_40_copper_proj_fe_skills_statistics_dev";
CREATE WIDGET TEXT source_schema DEFAULT "fe_skills_dev";                                       
CREATE WIDGET TEXT output_schema DEFAULT "output_202526_SN06"

In [0]:
/***********
National Provider Summary for Apprenticeships and Education & Training and community learning
Updated by:      Alison Cooper
Year:            2026
Update period:   Q1 August to January
Snapshot:        6 
Approx run time: 1-2 mins
Rows:			 
***********/
--** Note for 2024/25 - you will need to update the year in the final table at the end of of this code, as well the year and snapshot immediately below.** MT 19/12/2024

--update 09/12/25 include 4 years - 3 full plus latest


--Select required fields for latest 3 years
CREATE OR REPLACE TEMPORARY VIEW PARTICIPATION AS

SELECT 
  CASE 
    WHEN academic_year = ${academic_year} THEN
      CASE 
        WHEN ${snapshot} = 4 THEN CONCAT(academic_year, ' (Aug to Oct)')
        WHEN ${snapshot} = 6 THEN CONCAT(academic_year, ' (Aug to Jan)')
        WHEN ${snapshot} = 10 THEN CONCAT(academic_year, ' (Aug to Apr)')
        ELSE academic_year
      END
    ELSE academic_year
  END AS `year`,
provider_name,
cast (UKPRN as varchar(12)) as UKPRN,
sex,
lldd,
learner_home_depriv,
minority_ethnic,
fes_total,
apps_total,
eandt_total,
cl_total,
tl_total
FROM catalog_40_copper_proj_fe_skills_statistics_dev.fe_skills_dev.vw_learner_table
WHERE (snapshot = 14 AND  academic_year  IN (${academic_year} -303, ${academic_year} - 202, ${academic_year} - 101))
OR (snapshot = ${snapshot} AND  academic_year  = ${academic_year}) ;

In [0]:

--Calculate figures and format year* 
--*ie place a / (solidus) after the first 4 characters, so that year appears as, for example, 2022/23 rather than 202223

--National totals
CREATE OR REPLACE TEMPORARY VIEW NATIONAL AS

SELECT
1 as order_ref,
concat(substring(`year`,1,4) , '/' ,substring(`year`,5,22) ) as `year`,
'Total (All providers)' as provider_name,
'' as ukprn,
'Total' as category,
ifnull(sum(apps_total),0) as apps,
ifnull(sum(eandt_total),0)as et,
ifnull(sum(cl_total),0) as cl,
ifnull(sum(tl_total),0) as tl
FROM PARTICIPATION
GROUP BY `year`  ;

CREATE OR REPLACE TEMPORARY VIEW NATIONAL_SEX AS
SELECT
1 as order_ref,
concat(substring(`year`,1,4) , '/' ,substring(`year`,5,22) ) as `year`,
'Total (All providers)' as provider_name,
'' as ukprn,
case when sex = 'Female' then 'Sex - Female' when sex = 'Male' then 'Sex - Male' end as category,
ifnull(sum(apps_total),0) as apps,
ifnull(sum(eandt_total),0)as et,
ifnull(sum(cl_total),0) as cl,
ifnull(sum(tl_total),0) as tl
FROM PARTICIPATION
GROUP BY `year` , sex ;

CREATE OR REPLACE TEMPORARY VIEW NATIONAL_LLDD AS
SELECT
1 as order_ref,
concat(substring(`year`,1,4) , '/' ,substring(`year`,5,22) ) as `year`,
'Total (All providers)' as provider_name,
'' as ukprn,
case when lldd = 'With learning difficulty / disability' then 'LLDD - With learning difficulty / disability' when lldd = 'No learning difficulty / disability' then 'LLDD - No learning difficulty / disability' when lldd = 'Unknown' 
then 'LLDD - Unknown' end as category,
ifnull(sum(apps_total),0) as apps,
ifnull(sum(eandt_total),0)as et,
ifnull(sum(cl_total),0) as cl,
ifnull(sum(tl_total),0) as tl
FROM PARTICIPATION
GROUP BY `year` , lldd ;

CREATE OR REPLACE TEMPORARY VIEW NATIONAL_DEP AS
SELECT
1 as order_ref,
concat(substring(`year`,1,4) , '/' ,substring(`year`,5,22) ) as `year`,
'Total (All providers)' as provider_name,
'' as ukprn,
case when learner_home_depriv = 'Unknown' then 'IMD - Unknown' 
  when learner_home_depriv = 'One (most deprived)' then 'IMD - One (most deprived)' 
	when learner_home_depriv = 'Two' then 'IMD - Two'
	when learner_home_depriv = 'Three' then 'IMD - Three'
	when learner_home_depriv = 'Four' then 'IMD - Four'
	when learner_home_depriv = 'Five (least deprived)' then 'IMD - Five (least deprived)'
  else  learner_home_depriv end as category,
ifnull(sum(apps_total),0) as apps,
ifnull(sum(eandt_total),0)as et,
ifnull(sum(cl_total),0) as cl,
ifnull(sum(tl_total),0) as tl
FROM PARTICIPATION
GROUP BY `year`, learner_home_depriv ;


CREATE OR REPLACE TEMPORARY VIEW NATIONAL_ETH AS
SELECT
1 as order_ref,
concat(substring(`year`,1,4) , '/' ,substring(`year`,5,22) ) as `year`,
'Total (All providers)' as provider_name,
'' as ukprn,
case when minority_ethnic = 'White' then 'Ethnicity - White' when minority_ethnic = 'Unknown' then 'Ethnicity - Unknown' else minority_ethnic end as category,
ifnull(sum(apps_total),0) as apps,
ifnull(sum(eandt_total),0)as et,
ifnull(sum(cl_total),0) as cl,
ifnull(sum(tl_total),0) as tl
FROM PARTICIPATION
GROUP BY `year`, minority_ethnic ;


--Join national data together into one table
CREATE OR REPLACE TEMPORARY VIEW NATIONAL_TIDY AS
select *
from NATIONAL
union all
select *
from NATIONAL_SEX
union all
select *
from NATIONAL_LLDD
union all
select *
from NATIONAL_DEP
union all
select *
from NATIONAL_ETH  ;


--define detailed ordering for national data
CREATE OR REPLACE TEMPORARY VIEW NATIONAL_TIDY_ORDER AS
SELECT
order_ref,
case when provider_name =  'TOTAL (ALL PROVIDERS)' and ukprn = '' and category = 'Total' then 1
	 when category = 'Sex - Female' then 2
	 when category = 'Sex - Male' then 3
	 when category = 'LLDD - With learning difficulty / disability' then 4
	 when category = 'LLDD - No learning difficulty / disability' then 5
	 when category = 'LLDD - Unknown' then 6
	 when category = 'IMD - One (most deprived)' then 7
	 when category = 'IMD - Two' then 8
	 when category = 'IMD - Three' then 9
	 when category = 'IMD - Four' then 10
	 when category = 'IMD - Five (least deprived)' then 11
	 when category = 'IMD - Unknown' then 12
	 when category = 'Ethnicity - White' then 13
	 when category = 'Ethnic minorities (excluding white minorities)' then 14
	 when category = 'Ethnicity - Unknown' then 15
end as order_detailed,
`year`,
provider_name,
ukprn,
category,
apps,
et,
cl,
tl
FROM NATIONAL_TIDY
ORDER BY `year`, provider_name;


In [0]:
--Provider level
CREATE OR REPLACE TEMPORARY VIEW PROVIDER AS

SELECT
concat(substring(`year`,1,4) , '/' ,substring(`year`,5,22) ) as `year`,
provider_name,
UKPRN,
--apps
ifnull(sum(apps_total),0) as apps,
ifnull(sum(case when sex = 'Female' then apps_total end),0) as apps_female,
ifnull(sum(case when sex = 'Male' then apps_total end),0) as apps_male,
ifnull(sum(case when lldd = 'With learning difficulty / disability'  then apps_total end),0) as apps_lldd_yes,
ifnull(sum(case when lldd = 'No learning difficulty / disability' then apps_total end),0) as apps_lldd_no,
ifnull(sum(case when lldd = 'Unknown' then apps_total end),0) as apps_lldd_unknown,
ifnull(sum(case when learner_home_depriv = 'One (most deprived)' then apps_total end),0) as apps_dep_1,
ifnull(sum(case when learner_home_depriv = 'Two' then apps_total end),0) as apps_dep_2,
ifnull(sum(case when learner_home_depriv = 'Three' then apps_total end),0) as apps_dep_3,
ifnull(sum(case when learner_home_depriv = 'Four' then apps_total end),0) as apps_dep_4,
ifnull(sum(case when learner_home_depriv = 'Five (least deprived)' then apps_total end),0) as apps_dep_5,
ifnull(sum(case when learner_home_depriv = 'Unknown' then apps_total end),0) as apps_dep_u,
ifnull(sum(case when minority_ethnic = 'White' then apps_total end),0) as apps_white,
ifnull(sum(case when minority_ethnic = 'Ethnic minorities (excluding white minorities)' then apps_total end),0) as apps_ethnic_minorities,
ifnull(sum(case when minority_ethnic = 'Ethnicity - unknown' then apps_total end),0) as apps_ethnic_u,
--et
ifnull(sum(eandt_total),0) as et,
ifnull(sum(case when sex = 'Female' then eandt_total end),0) as et_female,
ifnull(sum(case when sex = 'Male' then eandt_total end),0) as et_male,
ifnull(sum(case when lldd = 'With learning difficulty / disability'  then eandt_total end),0) as et_lldd_yes,
ifnull(sum(case when lldd = 'No learning difficulty / disability' then eandt_total end),0) as et_lldd_no,
ifnull(sum(case when lldd = 'Unknown' then eandt_total end),0) as et_lldd_unknown,
ifnull(sum(case when learner_home_depriv = 'One (most deprived)' then eandt_total end),0) as et_dep_1,
ifnull(sum(case when learner_home_depriv = 'Two' then eandt_total end),0) as et_dep_2,
ifnull(sum(case when learner_home_depriv = 'Three' then eandt_total end),0) as et_dep_3,
ifnull(sum(case when learner_home_depriv = 'Four' then eandt_total end),0) as et_dep_4,
ifnull(sum(case when learner_home_depriv = 'Five (least deprived)' then eandt_total end),0) as et_dep_5,
ifnull(sum(case when learner_home_depriv = 'Unknown' then eandt_total end),0) as et_dep_u,
ifnull(sum(case when minority_ethnic = 'White' then eandt_total end),0) as et_white,
ifnull(sum(case when minority_ethnic = 'Ethnic minorities (excluding white minorities)' then eandt_total end),0) as et_ethnic_minorities,
ifnull(sum(case when minority_ethnic = 'Ethnicity unknown' then eandt_total end),0) as et_ethnic_u,
--cl
ifnull(sum(cl_total),0) as cl,
ifnull(sum(case when sex = 'Female' then cl_total end),0) as cl_female,
ifnull(sum(case when sex = 'Male' then cl_total end),0) as cl_male,
ifnull(sum(case when lldd = 'With learning difficulty / disability'  then cl_total end),0) as cl_lldd_yes,
ifnull(sum(case when lldd = 'No learning difficulty / disability' then cl_total end),0) as cl_lldd_no,
ifnull(sum(case when lldd = 'Unknown' then cl_total end),0) as cl_lldd_unknown,
ifnull(sum(case when learner_home_depriv = 'One (most deprived)' then cl_total end),0) as cl_dep_1,
ifnull(sum(case when learner_home_depriv = 'Two' then cl_total end),0) as cl_dep_2,
ifnull(sum(case when learner_home_depriv = 'Three' then cl_total end),0) as cl_dep_3,
ifnull(sum(case when learner_home_depriv = 'Four' then cl_total end),0) as cl_dep_4,
ifnull(sum(case when learner_home_depriv = 'Five (least deprived)' then cl_total end),0) as cl_dep_5,
ifnull(sum(case when learner_home_depriv = 'Unknown' then cl_total end),0) as cl_dep_u,
ifnull(sum(case when minority_ethnic = 'White' then cl_total end),0) as cl_white,
ifnull(sum(case when minority_ethnic = 'Ethnic minorities (excluding white minorities)' then cl_total end),0) as cl_ethnic_minorities,
ifnull(sum(case when minority_ethnic = 'Ethnicity unknown' then cl_total end),0) as cl_ethnic_u,
--tl
ifnull(sum(tl_total),0) as tl,
ifnull(sum(case when sex = 'Female' then tl_total end),0) as tl_female,
ifnull(sum(case when sex = 'Male' then tl_total end),0) as tl_male,
ifnull(sum(case when lldd = 'With learning difficulty / disability'  then tl_total end),0) as tl_lldd_yes,
ifnull(sum(case when lldd = 'No learning difficulty / disability' then tl_total end),0) as tl_lldd_no,
ifnull(sum(case when lldd = 'Unknown'  then tl_total end),0) as tl_lldd_unknown,
ifnull(sum(case when learner_home_depriv = 'One (most deprived)' then tl_total end),0) as tl_dep_1,
ifnull(sum(case when learner_home_depriv = 'Two' then tl_total end),0) as tl_dep_2,
ifnull(sum(case when learner_home_depriv = 'Three' then tl_total end),0) as tl_dep_3,
ifnull(sum(case when learner_home_depriv = 'Four' then tl_total end),0) as tl_dep_4,
ifnull(sum(case when learner_home_depriv = 'Five (least deprived)' then tl_total end),0) as tl_dep_5,
ifnull(sum(case when learner_home_depriv = 'Unknown' then tl_total end),0) as tl_dep_u,
ifnull(sum(case when minority_ethnic = 'White' then tl_total end),0) as tl_white,
ifnull(sum(case when minority_ethnic = 'Ethnic minorities (excluding white minorities)' then tl_total end),0) as tl_ethnic_minorities,
ifnull(sum(case when minority_ethnic = 'Ethnicity unknown' then tl_total end),0) as tl_ethnic_u

FROM PARTICIPATION
GROUP BY `year`, UKPRN, provider_name 
ORDER BY `year` desc, UKPRN, provider_name ;


--Put provider level data into a tidy format.
CREATE OR REPLACE TEMPORARY VIEW PROVIDER_TIDY AS

select `year`, provider_name, UKPRN, 'Total' as category, apps as apps, et as et, cl as cl, tl as tl
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'Sex - Female' as category, apps_female as apps, et_female as et, cl_female as cl, tl_female as tl
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'Sex - Male' as category, apps_male, et_male, cl_male, tl_male
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'LLDD - With learning difficulty / disability' as category, apps_lldd_yes, et_lldd_yes, cl_lldd_yes, tl_lldd_yes
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'LLDD - No learning difficulty / disability' as category, apps_lldd_no, et_lldd_no, cl_lldd_no, tl_lldd_no
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'LLDD - Unknown' as category, apps_lldd_unknown, et_lldd_unknown, cl_lldd_unknown, tl_lldd_unknown
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'IMD - One (most deprived)' as category, apps_dep_1, et_dep_1, cl_dep_1, tl_dep_1
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'IMD - Two' as category, apps_dep_2, et_dep_2, cl_dep_2, tl_dep_2
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'IMD - Three' as category, apps_dep_3, et_dep_3, cl_dep_3, tl_dep_3
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'IMD - Four' as category, apps_dep_4, et_dep_4, cl_dep_4,  tl_dep_4
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'IMD - Five (least deprived)' as category, apps_dep_5, et_dep_5, cl_dep_5, tl_dep_5
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'IMD - Unknown' as category, apps_dep_u, et_dep_u, cl_dep_u, tl_dep_u
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'Ethnicity - White' as category, apps_white, et_white, cl_white, tl_white
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'Ethnic minorities (excluding white minorities)' as category_type, apps_ethnic_minorities, et_ethnic_minorities, cl_ethnic_minorities, tl_ethnic_minorities
from PROVIDER
union all
select `year`, provider_name, UKPRN, 'Ethnicity - Unknown' as category, apps_ethnic_u, et_ethnic_u, cl_ethnic_u, tl_ethnic_u
from PROVIDER ;




In [0]:

--This identifies the providers with no participation, so they can be merged onto the file and then removed
CREATE OR REPLACE TEMPORARY VIEW PROVIDERS_WITH_ZERO_PARTICIPATION AS
select `year` , provider_name, UKPRN, 'Remove - all zeros' as providers_to_remove 
from PROVIDER
where apps = 0 and et = 0 and cl = 0 and tl = 0;

--Now merge this on to the provider file
CREATE OR REPLACE TEMPORARY VIEW PROVIDER_TIDY_WITH_PROVIDERS_WITH_ZERO_PARTICIPATION AS
SELECT 
PROVIDER_TIDY.*,
providers_to_remove
FROM PROVIDER_TIDY
LEFT JOIN PROVIDERS_WITH_ZERO_PARTICIPATION  ON PROVIDER_TIDY.`year`  = PROVIDERS_WITH_ZERO_PARTICIPATION.`year` 
																							AND PROVIDER_TIDY.provider_name = PROVIDERS_WITH_ZERO_PARTICIPATION.provider_name 
																							AND PROVIDER_TIDY.UKPRN = PROVIDERS_WITH_ZERO_PARTICIPATION.UKPRN		
																				
where providers_to_remove is null
;


--select * from PROVIDERS_WITH_ZERO_PARTICIPATION



In [0]:
--Define order of category_type field and general order for provider data
CREATE OR REPLACE TEMPORARY VIEW PROVIDER_TIDY_ORDER AS
SELECT
2 as order_ref,
case when provider_name <> 'TOTAL (ALL PROVIDERS)' and ukprn <> '' and category = 'Total' then 1
	 when category = 'Sex - Female' then 2
	 when category = 'Sex - Male' then 3
	 when category = 'LLDD - With learning difficulty / disability' then 4
	 when category = 'LLDD - No learning difficulty / disability' then 5
	 when category = 'LLDD - Unknown' then 6
	 when category = 'IMD - One (most deprived)' then 7
	 when category = 'IMD - Two' then 8
	 when category = 'IMD - Three' then 9
	 when category = 'IMD - Four' then 10
	 when category = 'IMD - Five (least deprived)' then 11
	 when category = 'IMD - Unknown' then 12
	 when category = 'Ethnicity - White' then 13
	 when category = 'Ethnic minorities (excluding white minorities)' then 14
	 when category = 'Ethnicity - Unknown' then 15
end as order_detailed,
`year`,
provider_name,
ukprn,
category,
apps,
et,
cl,
tl
FROM PROVIDER_TIDY_WITH_PROVIDERS_WITH_ZERO_PARTICIPATION 
ORDER BY `year`, provider_name ;

--Join national and provider level data together into one table

CREATE OR REPLACE TEMPORARY VIEW ALL_TIDY AS

select order_ref,order_detailed,`year` ,provider_name ,ukprn ,category  ,apps,et,cl,tl
from NATIONAL_TIDY_ORDER
union all
select order_ref,order_detailed,`year` ,provider_name ,ukprn ,category ,apps,et,cl,tl
from PROVIDER_TIDY_ORDER ;

In [0]:
--Produce final output
--Rounded and suppressed:
--replace values of 0,1,2,3,4 and 5 with 'low' and round remaining values to nearest 10.


CREATE OR REPLACE TEMPORARY VIEW ALL_FINAL_NEEDS_SORTING AS

SELECT
order_ref,
order_detailed,
--case when provider_name = 'TOTAL (ALL PROVIDERS)' then 1 else 2 end as provider_order,
`year`,
provider_name,
ukprn ,
category,
case when apps in (0,1,2,3,4) then 0 else round(apps,-1) end as apps,
case when et   in (0,1,2,3,4) then 0 else round(et,-1)   end as et,
case when cl   in (0,1,2,3,4) then 0 else round(cl ,-1)  end as cl,
case when tl   in (0,1,2,3,4) then 0 else round(tl ,-1)  end as tl
FROM ALL_TIDY

ORDER BY `year` desc, order_ref, provider_name, order_detailed  ;

CREATE OR REPLACE TEMPORARY VIEW ALL_FINAL AS
SELECT
--order_ref,
--order_detailed,
--case when provider_name = 'TOTAL (ALL PROVIDERS)' then 1 else 2 end as provider_order,
`year` as `Academic Year`,
provider_name as `Provider name`,
--CASE  WHEN provider_name != 'TOTAL (ALL PROVIDERS)' THEN CONCAT (provider_name,'_', ukprn) else provider_name end  as 'Provider name',
ukprn  As UKPRN,
category as `Learner characteristic`,
case when apps in (0,1,2,3,4) then 'low' else round(apps,-1) end as Apprenticeships,
case when et   in (0,1,2,3,4) then 'low' else round(et,-1)   end as `Education and Training`,
case when concat(cast(substring(cast(`year` as varchar(20)),1,4) as int),cast(substring(cast(`year` as varchar(20)),6,2) as int)) < 202425 then 'z' 
when tl = 0 then 'low' else tl end as `Tailored Learning`, 
case when concat(cast(substring(cast(`year` as varchar(20)),1,4) as int),cast(substring(cast(`year` as varchar(20)),6,2) as int)) > 202324 then 'z'
when cl = 0 then 'low' when cl > 4 then cl else 'z' end as `Community Learning` 
FROM ALL_FINAL_NEEDS_SORTING
ORDER BY `year` desc, order_ref, provider_name, order_detailed ;

SELECT * FROM ALL_FINAL

